<a href="https://colab.research.google.com/github/KatreenGhobrial/RepoCloudComputing/blob/main/HW/HW2/HW2_GIRAFFE_LAYERED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# My Basil Garden — Layered Project Notebook

The notebook is organized in a strict execution order:

1. **Setup & Configuration**
2. **Data Layer**
3. **Backend / Business Logic**
4. **UI Components**
5. **Frontend Application**
6. **QA Tests**
7. **Launch**

Run the cells from top to bottom. The application can work without a Gemini API key; AI-generated answers and image diagnosis are enabled only when a valid key is configured.


## 1. Setup & Configuration


In [1]:
# Install all runtime dependencies used by the notebook.
!pip -q install pandas numpy scikit-learn gradio nltk sentence-transformers markdown google-generativeai pillow matplotlib requests beautifulsoup4


In [2]:
import os
import re
import json
import time
import html as html_lib
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import gradio as gr
try:
    import markdown as markdown_lib
except ImportError:
    markdown_lib = None
import PIL.Image

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem import PorterStemmer

try:
    from sentence_transformers import SentenceTransformer
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    SentenceTransformer = None
    TRANSFORMERS_AVAILABLE = False

try:
    import google.generativeai as genai
    GEMINI_AVAILABLE = True
except ImportError:
    genai = None
    GEMINI_AVAILABLE = False


def render_markdown(text: str) -> str:
    if markdown_lib is not None:
        return markdown_lib.markdown(text)
    return html_lib.escape(str(text)).replace("\n", "<br>")


def load_gemini_api_key() -> Optional[str]:
    """Read the API key from Colab Secrets first, then from an environment variable."""
    key = os.getenv("GEMINI_API_KEY", "").strip()

    try:
        from google.colab import userdata
        colab_key = userdata.get("GEMINI_API_KEY")
        if colab_key:
            key = str(colab_key).strip()
    except Exception:
        pass

    return key or None


GEMINI_API_KEY = load_gemini_api_key()

if GEMINI_AVAILABLE and GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    vision_model = genai.GenerativeModel("gemini-2.5-flash")
else:
    vision_model = None

print("Gemini:", "configured" if vision_model else "not configured — local fallback remains available")
print("SentenceTransformers:", "available" if TRANSFORMERS_AVAILABLE else "not available — TF-IDF will be used")


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Gemini: not configured — local fallback remains available
SentenceTransformers: available


### API key configuration

In Google Colab, open **Secrets**, create a secret named `GEMINI_API_KEY`, and enable notebook access.  
Do not paste the key directly into the notebook.


## 2. Data Layer


In [3]:
basil_papers = [
       {
        "title": "Chemical Composition, Antioxidant and Antimicrobial Activities of Basil (Ocimum basilicum) Essential Oils Depends on Seasonal Variations",
        "authors": (
            "Abdullah Ijaz Hussain, Farooq Anwar, "
            "Syed Tufail Hussain Sherazi, Roman Przybylski"
        ),
        "journal": "Food Chemistry",
        "year": 2008,
        "doi": "10.1016/j.foodchem.2007.12.010",
        "url": "https://www.sciencedirect.com/science/article/abs/pii/S0308814607012666",
        "abstract": (
            "The study examined basil essential oils collected during summer, "
            "autumn, winter, and spring. Essential-oil yield, chemical "
            "composition, antioxidant activity, and antimicrobial activity "
            "changed significantly between seasons. Linalool was the dominant "
            "compound, while winter samples contained more oxygenated "
            "monoterpenes. The oils inhibited all tested bacterial and fungal "
            "microorganisms."
        )
    },

    {
        "title": "Basil Downy Mildew (Peronospora belbahrii): Discoveries and Challenges Relative to Its Control",
        "authors": (
            "Christian A. Wyenandt, James E. Simon, Robert M. Pyne, "
            "Kathryn Homa, Margaret T. McGrath, Shouan Zhang, "
            "Richard N. Raid, Li-Jun Ma, Robert Wick, Li Guo, "
            "Angela Madeiras"
        ),
        "journal": "Phytopathology",
        "year": 2015,
        "doi": "10.1094/PHYTO-02-15-0032-FI",
        "url": "https://doi.org/10.1094/PHYTO-02-15-0032-FI",
        "abstract": (
            "This review examines basil downy mildew caused by "
            "Peronospora belbahrii, a major threat to sweet basil production. "
            "The disease can spread through contaminated seed and is difficult "
            "to control because many commercial sweet basil varieties lack "
            "genetic resistance. The paper reviews breeding programs, pathogen "
            "biology, cultural management, and conventional and organic "
            "fungicide treatments."
        )
    },

    {
        "title": "Effects of Green Synthesized Zinc and Copper Nano-Fertilizers on the Morphological and Biochemical Attributes of Basil Plant",
        "authors": (
            "Ahmadreza Abbasifar, Fatemeh Shahrabadi, "
            "Babak ValizadehKaji"
        ),
        "journal": "Journal of Plant Nutrition",
        "year": 2020,
        "doi": "10.1080/01904167.2020.1724305",
        "url": "https://www.tandfonline.com/doi/full/10.1080/01904167.2020.1724305",
        "abstract": (
            "Zinc and copper nanoparticles were produced through green "
            "synthesis using basil extract and were applied to basil plants "
            "at different concentrations. Selected combinations improved "
            "plant growth, chlorophyll and carotenoid concentrations, phenolic "
            "and flavonoid contents, and antioxidant activity. The results "
            "show that the biological effects depend strongly on the "
            "concentrations of both nanoparticles."
        )
    },

    {
        "title": "Alteration in Light Spectra Causes Opposite Responses in Volatile Phenylpropanoids and Terpenoids Compared with Phenolic Acids in Sweet Basil (Ocimum basilicum) Leaves",
        "authors": (
            "Minna Kivimäenpää, Adedayo Mofikoya, "
            "Ahmed M. Abd El-Raheem, Johanna Riikonen, "
            "Riitta Julkunen-Tiitto, Jarmo K. Holopainen"
        ),
        "journal": "Journal of Agricultural and Food Chemistry",
        "year": 2022,
        "doi": "10.1021/acs.jafc.2c03309",
        "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9545148/",
        "abstract": (
            "Sweet basil plants were grown under three different LED light "
            "spectra to examine changes in growth, leaf anatomy, photosynthesis, "
            "and secondary metabolites. Light spectra that increased volatile "
            "terpenoids and phenylpropanoids tended to reduce phenolic acids "
            "and plant growth, while spectra that promoted growth and phenolic "
            "acids reduced several volatile compounds. Glandular trichome "
            "density was associated with terpenoid and eugenol production."
        )
    },
   {
    "title": "First Report of Bacterial Leaf Spot Caused by Pseudomonas cichorii on Thai Basil in Hawaii",
    "authors": (
        "Blaine C. Luiz, Wade P. Heller, Eva Brill, "
        "Brian C. Bushe, Lisa M. Keith"
    ),
    "journal": "Plant Disease",
    "year": 2018,
    "doi": "10.1094/PDIS-06-18-0995-PDN",
    "url": "https://apsjournals.apsnet.org/doi/10.1094/PDIS-06-18-0995-PDN",
    "abstract": (
        "Water-soaked, irregular, gray-to-black leaf spots were observed "
        "on Thai basil plants. The lesions ranged from small spots to larger "
        "necrotic areas near the leaf margins. Laboratory isolation and "
        "pathogenicity testing identified Pseudomonas cichorii as the cause "
        "of the bacterial leaf spot disease."
    )
}
]


In [4]:
STOP_WORDS = {
    'a', 'about', 'above', 'after', 'again', 'against', 'all', 'am',
    'an', 'and', 'any', 'are', 'aren', 'as', 'at', 'be', 'because',
    'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by',
    'can', 'could', 'couldn', 'did', 'didn', 'do', 'does', 'doesn',
    'doing', 'don', 'down', 'during', 'each', 'few', 'for', 'from',
    'further', 'had', 'hadn', 'has', 'hasn', 'have', 'haven', 'having',
    'he', 'her', 'here', 'hers', 'herself', 'him', 'himself', 'his',
    'how', 'i', 'if', 'in', 'into', 'is', 'isn', 'it', 'its', 'itself',
    'just', 'll', 'm', 'ma', 'me', 'might', 'more', 'most', 'mustn',
    'my', 'myself', 'needn', 'no', 'nor', 'not', 'now', 'of', 'off',
    'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves',
    'out', 'over', 'own', 're', 's', 'same', 'shan', 'she', 'should',
    'shouldn', 'so', 'some', 'such', 't', 'than', 'that', 'the',
    'their', 'theirs', 'them', 'themselves', 'then', 'there', 'these',
    'they', 'this', 'those', 'through', 'to', 'too', 'under', 'until',
    'up', 've', 'very', 'was', 'wasn', 'we', 'were', 'weren', 'what',
    'when', 'where', 'which', 'while', 'who', 'whom', 'why', 'will',
    'with', 'won', 'would', 'wouldn', 'y', 'you', 'your', 'yours',
    'yourself', 'yourselves'
}

TARGET_TERMS = {
    'terpenoids', 'phenolic', 'spectra', 'trichome', 'nanoparticles',
    'chlorophyll', 'flavonoid', 'antioxidant', 'peronospora', 'mildew',
    'seed', 'fungicide', 'essential', 'antimicrobial', 'linalool',
    'composition', 'pseudomonas', 'spot', 'lesion', 'necrotic'
}

stemmer = PorterStemmer()


def build_target_inverted_index(
    papers: List[Dict[str, Any]],
    target_terms: set[str] = TARGET_TERMS
) -> Dict[str, Dict[str, Any]]:
    """Build the tutorial-style term index: total count and source URLs."""
    index: Dict[str, Dict[str, Any]] = {}
    target_stems = {stemmer.stem(term): term for term in target_terms}

    for paper in papers:
        text = f"{paper.get('title', '')} {paper.get('abstract', '')}".lower()
        for word in re.findall(r"\w+", text):
            if word in STOP_WORDS:
                continue

            stemmed_word = stemmer.stem(word)
            if word in target_terms or stemmed_word in target_stems:
                saved_term = word if word in target_terms else target_stems[stemmed_word]
                item = index.setdefault(saved_term, {"count": 0, "DocIDs": set()})
                item["count"] += 1
                item["DocIDs"].add(paper.get("url", ""))

    return index


inverted_index = build_target_inverted_index(basil_papers)

print("=== TARGET TERM INDEX ===")
for term in sorted(inverted_index):
    item = inverted_index[term]
    print(f"{term:15} | count={item['count']:2} | documents={len(item['DocIDs'])}")


=== TARGET TERM INDEX ===
antimicrobial   | count= 2 | documents=1
antioxidant     | count= 3 | documents=2
chlorophyll     | count= 1 | documents=1
composition     | count= 2 | documents=1
essential       | count= 3 | documents=1
flavonoid       | count= 1 | documents=1
fungicide       | count= 1 | documents=1
lesion          | count= 1 | documents=1
linalool        | count= 1 | documents=1
mildew          | count= 2 | documents=1
nanoparticles   | count= 2 | documents=1
necrotic        | count= 1 | documents=1
peronospora     | count= 2 | documents=1
phenolic        | count= 4 | documents=2
pseudomonas     | count= 2 | documents=1
seed            | count= 1 | documents=1
spectra         | count= 4 | documents=1
spot            | count= 4 | documents=1
terpenoids      | count= 3 | documents=1
trichome        | count= 1 | documents=1


## 3. Backend / Business Logic


In [5]:
class SimpleVectorStore:
    """Small in-memory vector store used by the RAG service."""

    def __init__(self):
        self.documents: List[str] = []
        self.embeddings: List[Any] = []
        self.metadatas: List[Dict[str, Any]] = []
        self.ids: List[str] = []

    def clear(self) -> None:
        self.documents.clear()
        self.embeddings.clear()
        self.metadatas.clear()
        self.ids.clear()

    def add(self, embeddings, documents, metadatas, ids) -> None:
        self.embeddings.extend(np.asarray(embeddings))
        self.documents.extend(documents)
        self.metadatas.extend(metadatas)
        self.ids.extend(ids)

    def query(self, query_embeddings, n_results: int = 3) -> Dict[str, List[List[Any]]]:
        if not self.embeddings:
            return {
                "ids": [[]],
                "documents": [[]],
                "metadatas": [[]],
                "distances": [[]],
            }

        similarities = cosine_similarity(
            np.asarray(query_embeddings),
            np.asarray(self.embeddings)
        )[0]

        result_count = min(max(int(n_results), 1), len(self.documents))
        top_indices = np.argsort(similarities)[::-1][:result_count]

        return {
            "ids": [[self.ids[i] for i in top_indices]],
            "documents": [[self.documents[i] for i in top_indices]],
            "metadatas": [[self.metadatas[i] for i in top_indices]],
            "distances": [[float(1 - similarities[i]) for i in top_indices]],
        }


class EcologicalRAG:
    """Retrieval service with optional Gemini answer generation."""

    def __init__(
        self,
        gemini_api_key: Optional[str] = None,
        use_transformers: bool = True
    ):
        self.collection = SimpleVectorStore()
        self.papers: List[Dict[str, Any]] = []
        self.fitted = False
        self.use_transformers = False
        self.embedding_model = None
        self.tfidf = TfidfVectorizer(max_features=1000, stop_words="english")

        if use_transformers and TRANSFORMERS_AVAILABLE:
            try:
                self.embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
                self.use_transformers = True
            except Exception as exc:
                print(f"SentenceTransformer fallback to TF-IDF: {exc}")

        self.use_gemini = bool(gemini_api_key and GEMINI_AVAILABLE)
        self.text_model = None

        if self.use_gemini:
            genai.configure(api_key=gemini_api_key)
            self.text_model = genai.GenerativeModel("gemini-2.5-flash")

    @staticmethod
    def preprocess_text(text: str) -> str:
        if not text:
            return ""
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r"[^\w\s\-\.]", " ", text)
        return text.strip()

    def generate_embeddings(self, texts: List[str]):
        if self.use_transformers:
            return self.embedding_model.encode(texts, show_progress_bar=False)

        if not self.fitted:
            self.tfidf.fit(texts)
            self.fitted = True

        return self.tfidf.transform(texts).toarray()

    def load_papers(self, papers_data: List[Dict[str, Any]]) -> None:
        valid_papers = [p for p in papers_data if p.get("abstract", "").strip()]
        if not valid_papers:
            raise ValueError("No papers with abstracts were supplied.")

        documents = [
            self.preprocess_text(f"{paper.get('title', '')}. {paper.get('abstract', '')}")
            for paper in valid_papers
        ]
        metadatas = [
            {
                "title": paper.get("title", "Unknown"),
                "authors": paper.get("authors", "Unknown"),
                "url": paper.get("url", ""),
            }
            for paper in valid_papers
        ]
        ids = [f"paper_{i}" for i in range(len(valid_papers))]

        self.collection.clear()
        self.papers = list(valid_papers)

        # TF-IDF must be refitted when the document collection changes.
        self.fitted = False
        self.tfidf = TfidfVectorizer(max_features=1000, stop_words="english")
        embeddings = self.generate_embeddings(documents)

        self.collection.add(
            embeddings=embeddings,
            documents=documents,
            metadatas=metadatas,
            ids=ids,
        )

    def search(self, query: str, n_results: int = 3):
        query_processed = self.preprocess_text(query)
        if not query_processed:
            return {
                "ids": [[]],
                "documents": [[]],
                "metadatas": [[]],
                "distances": [[]],
            }

        query_embedding = self.generate_embeddings([query_processed])
        return self.collection.query(query_embedding, n_results=n_results)

    def _local_grounded_response(
        self,
        query_text: str,
        relevant_papers: List[Dict[str, Any]]
    ) -> str:
        if not relevant_papers:
            return "No relevant paper was found in the local knowledge base."

        lines = [
            f"**Local research summary for:** {query_text}",
            "",
            "The most relevant papers in the current knowledge base are:",
        ]
        for paper in relevant_papers:
            lines.append(
                f"- **{paper['title']}** — {paper['abstract'][:280].strip()}..."
            )
        lines.append("")
        lines.append("_Gemini is not configured, so this answer is a retrieval-based local summary._")
        return "\n".join(lines)

    def query(self, query_text: str, n_results: int = 3) -> Dict[str, Any]:
        search_results = self.search(query_text, n_results)
        relevant_papers = []

        for doc_id in search_results["ids"][0]:
            paper_index = int(doc_id.split("_")[1])
            relevant_papers.append(self.papers[paper_index])

        if not self.use_gemini:
            response_text = self._local_grounded_response(query_text, relevant_papers)
        else:
            context_parts = []
            for i, paper in enumerate(relevant_papers):
                document = search_results["documents"][0][i]
                context_parts.append(
                    f"Paper: {paper['title']}\n"
                    f"Authors: {paper['authors']}\n"
                    f"Source: {paper['url']}\n"
                    f"Content: {document}"
                )

            prompt = (
                "You are an agriculture and basil-plant research assistant. "
                "Answer only from the supplied research context. "
                "If the context does not support a claim, say so explicitly.\n\n"
                f"Question: {query_text}\n\n"
                f"Research context:\n{'\n\n'.join(context_parts)}"
            )

            try:
                response = self.text_model.generate_content(
                    prompt,
                    generation_config=genai.types.GenerationConfig(
                        temperature=0.3,
                        max_output_tokens=800,
                    ),
                )
                response_text = response.text
            except Exception as exc:
                response_text = (
                    f"Gemini request failed: {exc}\n\n"
                    + self._local_grounded_response(query_text, relevant_papers)
                )

        return {
            "response": response_text,
            "papers_found": len(relevant_papers),
            "papers": relevant_papers,
            "search_results": search_results,
        }


rag_system = EcologicalRAG(
    gemini_api_key=GEMINI_API_KEY,
    use_transformers=False
)
rag_system.load_papers(basil_papers)

print(f"RAG service ready with {len(rag_system.papers)} papers.")


RAG service ready with 5 papers.


In [6]:
def build_paper_index(paper: Dict[str, Any]) -> Dict[str, int]:
    index: Dict[str, int] = {}
    text = f"{paper.get('title', '')} {paper.get('abstract', '')}".lower()

    for word in re.findall(r"\w+", text):
        if word in STOP_WORDS:
            continue
        stemmed_word = stemmer.stem(word)
        index[stemmed_word] = index.get(stemmed_word, 0) + 1

    return index


def prepare_query_words(query: str) -> List[str]:
    return [
        stemmer.stem(word)
        for word in re.findall(r"\w+", query.lower())
        if word not in STOP_WORDS and word not in {"and", "or"}
    ]


def boolean_search(query: str, mode: str = "AUTO", limit: int = 3) -> List[Dict[str, Any]]:
    query = query.strip()
    if not query:
        return []

    normalized_mode = mode.upper()
    if normalized_mode == "AUTO":
        lowered = f" {query.lower()} "
        normalized_mode = "AND" if " and " in lowered else "OR"

    clean_query = re.sub(r"\b(?:AND|OR)\b", " ", query, flags=re.IGNORECASE)
    query_words = prepare_query_words(clean_query)
    if not query_words:
        return []

    matched_papers = []

    for paper in basil_papers:
        paper_index = build_paper_index(paper)
        found_words = {
            word: paper_index[word]
            for word in query_words
            if word in paper_index
        }

        is_match = (
            len(found_words) == len(set(query_words))
            if normalized_mode == "AND"
            else bool(found_words)
        )

        if is_match:
            result = dict(paper)
            result["found_words"] = found_words
            result["score"] = sum(found_words.values())
            result["mode"] = normalized_mode
            matched_papers.append(result)

    matched_papers.sort(key=lambda item: item["score"], reverse=True)
    return matched_papers[:limit]


def advanced_html_search(query: str, mode: str = "AUTO"):
    """Generator used by Gradio to show a loading state and then final results."""
    if not query or not query.strip():
        yield "<div style='color:#ef4444; padding:20px;'>Please enter a search term.</div>"
        return

    yield """
    <div style="text-align:center; padding:40px;">
        <div style="font-size:40px; margin-bottom:10px;">🔍</div>
        <h3 style="color:#7edf62;">Scanning Knowledge Base...</h3>
    </div>
    """

    matched_papers = boolean_search(query, mode)
    effective_mode = matched_papers[0]["mode"] if matched_papers else (
        "AND" if " and " in f" {query.lower()} " else "OR"
    )

    if not matched_papers:
        papers_html = f"""
        <div style="text-align:center; padding:30px; color:#aabd9e;">
            <h3>No exact Boolean Search matches for “{html_lib.escape(query)}”.</h3>
            <p>The semantic RAG search was still executed.</p>
        </div>
        """
    else:
        papers_html = (
            f"<div style='color:#9cb896; margin-bottom:20px;'>"
            f"Found {len(matched_papers)} Boolean result(s), mode: <b>{effective_mode}</b>.</div>"
        )

        for paper in matched_papers:
            found_words_text = ", ".join(
                f"{html_lib.escape(word)}: {count}"
                for word, count in paper["found_words"].items()
            )
            papers_html += f"""
            <div style="background:rgba(26,38,20,.4); border:1px solid rgba(74,124,46,.2);
                        border-radius:12px; padding:20px; margin-bottom:16px;">
                <div style="font-size:18px; font-weight:700; color:#c2eaaf;">
                    {html_lib.escape(paper.get("title", "Unknown"))}
                </div>
                <div style="font-size:13px; color:#9cb896; margin:8px 0 12px;">
                    {html_lib.escape(paper.get("authors", "Unknown"))}
                </div>
                <div style="font-size:14px; color:#d1d5db; line-height:1.6;">
                    {html_lib.escape(paper.get("abstract", ""))}
                </div>
                <div style="margin-top:14px; color:#fbbf24; font-size:12px;">
                    Score: {paper["score"]} · Found: {found_words_text}
                </div>
                <div style="margin-top:12px;">
                    <a href="{html_lib.escape(paper.get('url', '#'))}" target="_blank"
                       style="color:#7edf62;">Open source ↗</a>
                </div>
            </div>
            """

    yield """
    <div style="text-align:center; padding:30px; color:#fbbf24;">
        <div style="font-size:32px;">🤖 ⏳</div>
        <h3>Generating a grounded summary...</h3>
    </div>
    """ + papers_html

    rag_result = rag_system.query(query, n_results=3)
    ai_summary = render_markdown(rag_result["response"])

    final_html = f"""
    <div style="background:linear-gradient(135deg, rgba(26,38,20,.8), rgba(15,26,10,.9));
                border:1px solid rgba(245,158,11,.3); border-radius:16px;
                padding:24px; margin-bottom:24px;">
        <h3 style="color:#fbbf24; margin-top:0;">✨ AI-Enhanced Summary</h3>
        <div style="color:#e8f0e4; font-size:15px; line-height:1.7;">{ai_summary}</div>
    </div>
    """

    yield final_html + papers_html


def search_and_wrap(query: str):
    for step in advanced_html_search(query):
        yield f"<div class='results-container'>{step}</div>"


In [7]:
# Image upload and AI scanner service

def get_uploaded_file_path(file_obj: Any) -> Optional[str]:
    if file_obj is None:
        return None
    if isinstance(file_obj, str):
        return file_obj
    for attribute in ("name", "path"):
        value = getattr(file_obj, attribute, None)
        if value:
            return value
    return None


def update_upload_preview(files):
    files = files or []
    paths = [path for path in (get_uploaded_file_path(item) for item in files) if path]

    if not paths:
        return (
            """
            <div class="page-card" style="text-align:center;">
                <div style="font-size:40px;">📷</div>
                <div style="color:white; font-size:18px; font-weight:800;">No images uploaded</div>
                <div style="color:#aabd9e;">Upload one or more basil images to begin.</div>
            </div>
            """,
            [],
        )

    return (
        f"""
        <div class="page-card">
            <div style="color:#30d158; font-weight:800;">✅ {len(paths)} image(s) ready for analysis</div>
        </div>
        """,
        paths,
    )


def run_gemini_analysis(files):
    files = files or []
    if not files:
        return "<div style='color:#ff453a; padding:10px;'>⚠️ Upload at least one image first.</div>"

    if vision_model is None:
        return (
            "<div class='page-card' style='color:#ffcc00;'>"
            "Gemini image analysis is disabled. Add `GEMINI_API_KEY` to Colab Secrets and rerun Setup."
            "</div>"
        )

    reports = []

    for index, file_obj in enumerate(files, start=1):
        try:
            image_path = get_uploaded_file_path(file_obj)
            if not image_path:
                raise ValueError("The uploaded file path could not be read.")

            image = PIL.Image.open(image_path)
            filename = os.path.basename(image_path)
            prompt = (
                "You are an expert plant pathologist. Analyze this basil image. "
                "Return only valid JSON with keys: "
                "status (healthy, mild, or severe), diagnosis (short string), "
                "and recommendations (list of exactly three actionable strings)."
            )

            response = vision_model.generate_content([prompt, image])
            clean_text = response.text.strip().replace("```json", "").replace("```", "")
            result = json.loads(clean_text)

            status = str(result.get("status", "unknown")).lower()
            diagnosis = html_lib.escape(str(result.get("diagnosis", "No diagnosis provided.")))
            recommendations = result.get("recommendations", [])
            if not isinstance(recommendations, list):
                recommendations = [str(recommendations)]

            status_styles = {
                "healthy": ("✅ Healthy", "#30d158", "rgba(48,209,88,.4)"),
                "mild": ("🟡 Mild issue", "#ffcc00", "rgba(255,204,0,.4)"),
                "severe": ("🔴 Severe issue", "#ff453a", "rgba(255,69,58,.4)"),
            }
            status_label, status_color, border_color = status_styles.get(
                status, ("⚪ Unknown", "#aabd9e", "rgba(170,189,158,.4)")
            )

            recommendations_html = "".join(
                f"<li>{html_lib.escape(str(item))}</li>"
                for item in recommendations[:3]
            )

            reports.append(f"""
            <div class="page-card" style="margin-top:14px; border:1px solid {border_color};">
                <div style="display:flex; justify-content:space-between; gap:12px;">
                    <strong style="color:white;">{html_lib.escape(filename)}</strong>
                    <span style="color:{status_color}; font-weight:800;">{status_label}</span>
                </div>
                <p style="color:#e8f0e4;"><strong>Diagnosis:</strong> {diagnosis}</p>
                <div style="color:#aabd9e;"><strong>Recommendations:</strong>
                    <ul>{recommendations_html}</ul>
                </div>
            </div>
            """)
        except Exception as exc:
            reports.append(
                f"<div style='color:#ff453a; padding:10px;'>"
                f"⚠️ Image #{index} failed: {html_lib.escape(str(exc))}</div>"
            )

    return f"""
    <div class="page-card" style="margin-top:24px;">
        <h3 style="color:#7edf62;">🧬 AI Batch Diagnostics Report</h3>
        {''.join(reports)}
    </div>
    """


In [8]:
# Sensor data service

BASE_URL = "https://server-cloud-v645.onrender.com"


def get_sensor_data(feed: str, limit: int) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    try:
        response = requests.get(
            f"{BASE_URL}/history",
            params={"feed": feed, "limit": int(limit)},
            timeout=15,
        )
        response.raise_for_status()
        payload = response.json()
    except Exception as exc:
        return pd.DataFrame(), {"error": str(exc)}

    rows = payload.get("data", [])
    if not isinstance(rows, list):
        return pd.DataFrame(), {"error": "The server returned an invalid data format."}

    df = pd.DataFrame(rows)
    if "created_at" in df.columns:
        df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
        df = df.sort_values("created_at")
    if "value" in df.columns:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")

    return df, payload


def sensor_status(feed: str, value: Any) -> Tuple[str, str]:
    try:
        numeric_value = float(value)
    except (TypeError, ValueError):
        return "No Data", "#aabd9e"

    if feed == "temperature":
        if numeric_value < 18:
            return "Low", "#4d8dff"
        if numeric_value <= 30:
            return "Optimal", "#30d158"
        return "High", "#ff453a"

    if feed == "humidity":
        if numeric_value < 40:
            return "Low", "#ffcc00"
        if numeric_value <= 70:
            return "Optimal", "#30d158"
        return "High", "#4d8dff"

    if feed == "soil":
        if numeric_value < 25:
            return "Low", "#ff453a"
        if numeric_value <= 80:
            return "Optimal", "#30d158"
        return "Too Wet", "#ffcc00"

    return "Unknown", "#aabd9e"


def check_manager_alerts(df: pd.DataFrame, feed: str) -> str:
    if df.empty or "value" not in df.columns:
        return "⚠️ No valid reading is available."

    values = df["value"].dropna()
    if values.empty:
        return "⚠️ No numeric reading is available."

    latest = float(values.iloc[-1])
    status, _ = sensor_status(feed, latest)

    messages = {
        ("temperature", "Low"): "❄️ Temperature is too low for basil. Move the plant to a warmer area.",
        ("temperature", "Optimal"): "✅ Temperature is within the recommended basil range.",
        ("temperature", "High"): "🔥 Temperature is too high. Add shade and check ventilation.",
        ("humidity", "Low"): "💧 Air humidity is low. Consider gentle misting or a humidifier.",
        ("humidity", "Optimal"): "✅ Air humidity is within the preferred range.",
        ("humidity", "High"): "🌫️ Air humidity is high. Improve airflow to reduce disease risk.",
        ("soil", "Low"): "🚿 Soil moisture is low. Check whether the basil needs watering.",
        ("soil", "Optimal"): "✅ Soil moisture is within the preferred range.",
        ("soil", "Too Wet"): "⚠️ Soil is too wet. Pause watering and check drainage.",
    }
    return messages.get((feed, status), f"Status: {status}")


def get_latest_value_for_card(feed: str, default_value: Any = "--"):
    df, _ = get_sensor_data(feed, 10)
    if df.empty or "value" not in df.columns:
        return default_value

    values = df["value"].dropna()
    return round(float(values.iloc[-1]), 2) if not values.empty else default_value


def update_live_sensor_screen(feed: str, limit: int):
    df, raw_data = get_sensor_data(feed, limit)

    if df.empty or "value" not in df.columns or df["value"].dropna().empty:
        error_message = html_lib.escape(str(raw_data.get("error", raw_data)))
        return (
            f"<div class='page-card' style='color:#ff453a;'>Sensor error: {error_message}</div>",
            None,
            df,
        )

    latest_value = float(df["value"].dropna().iloc[-1])
    manager_alert = check_manager_alerts(df, feed)

    html_result = f"""
    <div class="main-wrapper">
        <div class="page-card">
            <div class="page-title">{feed.title()} history</div>
            <div class="page-subtitle">Latest: {latest_value:.2f} · Samples: {int(limit)}</div>
            <div style="margin-top:18px; padding:14px; background:rgba(0,0,0,.3);
                        border:1px solid rgba(126,214,98,.2); border-radius:12px;">
                {manager_alert}
            </div>
        </div>
    </div>
    """

    fig, ax = plt.subplots(figsize=(10, 4))
    if "created_at" in df.columns:
        ax.plot(df["created_at"], df["value"], marker="o")
        ax.set_xlabel("Time")
    else:
        ax.plot(df.index, df["value"], marker="o")
        ax.set_xlabel("Sample")
    ax.set_ylabel("Value")
    ax.set_title(f"{feed.title()} history")
    plt.xticks(rotation=30)
    plt.tight_layout()

    return html_result, fig, df


def make_sensor_cards_html(fetch_live: bool = True):
    values = (
        {
            "temperature": get_latest_value_for_card("temperature"),
            "humidity": get_latest_value_for_card("humidity"),
            "soil": get_latest_value_for_card("soil"),
        }
        if fetch_live
        else {"temperature": "--", "humidity": "--", "soil": "--"}
    )

    cards = []
    specs = [
        ("temperature", "🌡️", "Temperature", "°C"),
        ("humidity", "💧", "Air Humidity", "%"),
        ("soil", "🌱", "Soil Moisture", "%"),
    ]

    for feed, icon, label, unit in specs:
        value = values[feed]
        status, color = sensor_status(feed, value)
        cards.append(f"""
        <div class="page-card" style="text-align:center;">
            <div style="font-size:30px;">{icon}</div>
            <div style="font-size:28px; font-weight:900; color:white;">{value}{unit if value != '--' else ''}</div>
            <div style="font-size:14px; font-weight:800; color:white;">{label}</div>
            <div style="color:{color}; font-weight:800;">{status}</div>
        </div>
        """)

    return f"""
    <div class="main-wrapper">
        <div class="page-card" style="margin-bottom:24px;">
            <div class="page-title">Live Telemetry</div>
            <div class="page-subtitle">Readings from the Render sensor service</div>
        </div>
        <div style="display:grid; grid-template-columns:repeat(3,1fr); gap:18px;">
            {''.join(cards)}
        </div>
    </div>
    """


def make_dashboard_sensor_summary_html(fetch_live: bool = True):
    values = (
        {
            "temperature": get_latest_value_for_card("temperature"),
            "humidity": get_latest_value_for_card("humidity"),
            "soil": get_latest_value_for_card("soil"),
        }
        if fetch_live
        else {"temperature": "--", "humidity": "--", "soil": "--"}
    )

    return f"""
    <div class="main-wrapper">
        <div class="page-card" style="margin:24px 0;">
            <div style="font-size:20px; font-weight:900; color:white;">Sensor Summary</div>
            <div style="display:grid; grid-template-columns:repeat(3,1fr); gap:14px; margin-top:18px;">
                <div class="info-mini-card">🌡️ <b>Temperature:</b> {values['temperature']}°C</div>
                <div class="info-mini-card">💧 <b>Humidity:</b> {values['humidity']}%</div>
                <div class="info-mini-card">🌱 <b>Soil:</b> {values['soil']}%</div>
            </div>
        </div>
    </div>
    """


def show_sensor_graph(feed: str, limit: int):
    html_result, fig, df = update_live_sensor_screen(feed, limit)
    return make_sensor_cards_html(), html_result, fig, df


def show_temperature_graph(limit):
    return show_sensor_graph("temperature", limit)


def show_humidity_graph(limit):
    return show_sensor_graph("humidity", limit)


def show_soil_graph(limit):
    return show_sensor_graph("soil", limit)


## 4. UI Components


In [9]:
# ==========================================
# CELL 5: Frontend UI & Application Launch
# ==========================================

GLOBAL_CSS = """
body { background: #050d05 !important; margin: 0; }
.gradio-container { background: radial-gradient(circle at top, #0a1f0a 0%, #050d05 70%) !important; color: #f5f7f0 !important; font-family: 'Inter', system-ui, sans-serif !important; }
.main-wrapper { max-width: 1200px; margin: auto; padding: 10px; }

/* --- GLASSMORPHISM CARDS --- */
.page-card, .metric-card {
    background: rgba(15, 35, 15, 0.45) !important;
    backdrop-filter: blur(12px) !important;
    border: 1px solid rgba(126, 214, 98, 0.15) !important;
    border-radius: 20px;
    padding: 24px;
    box-shadow: 0 8px 32px rgba(0, 0, 0, 0.2);
    transition: transform 0.2s ease, box-shadow 0.2s ease;
}
.metric-card:hover { transform: translateY(-4px); box-shadow: 0 12px 40px rgba(0, 0, 0, 0.3); border-color: rgba(126, 214, 98, 0.3) !important; }
.metric-card { padding: 20px; min-height: 120px; display: flex; flex-direction: column; justify-content: center; }

/* --- HEADER & NAVBAR --- */
.app-header { display: flex; justify-content: space-between; align-items: center; padding: 10px 20px; margin-bottom: 20px; background: rgba(10, 25, 10, 0.6); backdrop-filter: blur(10px); border: 1px solid rgba(126, 214, 98, 0.1); border-radius: 20px; }
.logo-box { display: flex; align-items: center; gap: 14px; }
.logo-icon { width: 44px; height: 44px; border-radius: 14px; background: linear-gradient(135deg, #246e24, #123b16); display: flex; align-items: center; justify-content: center; font-size: 24px; box-shadow: 0 4px 15px rgba(36, 110, 36, 0.4); }
.logo-title { font-size: 22px; font-weight: 800; color: #ffffff; letter-spacing: -0.5px; }
.logo-subtitle { font-size: 13px; color: #8bb381; font-weight: 500; }

.nav-row { background: transparent !important; gap: 10px !important; }
.nav-btn button { background: rgba(255, 255, 255, 0.05) !important; color: #aabd9e !important; border: 1px solid rgba(255, 255, 255, 0.05) !important; border-radius: 12px !important; font-weight: 600 !important; font-size: 14px !important; min-height: 44px !important; transition: all 0.3s cubic-bezier(0.4, 0, 0.2, 1) !important; box-shadow: none !important; }
.nav-btn button:hover { background: rgba(126, 214, 98, 0.15) !important; color: #ffffff !important; border-color: rgba(126, 214, 98, 0.4) !important; transform: scale(1.02); }

/* --- TYPOGRAPHY & LAYOUT --- */
.page-title { font-size: 30px; font-weight: 800; color: #ffffff; margin-bottom: 6px; letter-spacing: -0.5px; }
.page-subtitle { color: #aabd9e; font-size: 15px; }
.grid-4 { display: grid; grid-template-columns: repeat(4, 1fr); gap: 16px; margin-bottom: 24px; }
.grid-2 { display: grid; grid-template-columns: 2fr 1fr; gap: 20px; }

/* --- SEARCH BAR CUSTOMIZATION --- */
.search-bar-row { background: rgba(15, 35, 15, 0.6) !important; border: 1px solid rgba(126, 214, 98, 0.3) !important; border-radius: 20px !important; padding: 12px 14px !important; align-items: flex-start !important; margin: 0 auto 16px auto !important; max-width: 800px !important; box-shadow: 0 8px 25px rgba(0,0,0,0.2) !important; transition: border-color 0.3s ease; }
.search-bar-row:focus-within { border-color: #7edf62 !important; box-shadow: 0 8px 30px rgba(126, 214, 98, 0.15) !important; }
.search-bar-row .gr-box, .search-bar-row label { background: transparent !important; border: none !important; box-shadow: none !important; }
.search-bar-row textarea { background: transparent !important; border: none !important; box-shadow: none !important; font-size: 15px !important; color: #fff !important; padding-left: 10px !important; resize: none !important; overflow-y: auto !important; }
.search-btn-custom { border-radius: 12px !important; padding: 10px 30px !important; margin-top: 10px !important; background: linear-gradient(135deg, #3a7d2a, #2d5a20) !important; border: none !important; font-size: 15px !important; color: white !important;}
.search-btn-custom:hover { background: linear-gradient(135deg, #4e9e38, #3a7d2a) !important; }

/* --- RESULTS AREA SCROLL --- */
.results-container { max-height: 550px; overflow-y: auto; padding-right: 10px; }
.results-container::-webkit-scrollbar { width: 6px; }
.results-container::-webkit-scrollbar-track { background: rgba(15, 35, 15, 0.4); border-radius: 10px; }
.results-container::-webkit-scrollbar-thumb { background: rgba(126, 214, 98, 0.4); border-radius: 10px; }

/* --- CHIPS & TOGGLES --- */
.green-pill { display: inline-block; padding: 6px 14px; border-radius: 999px; background: rgba(126, 214, 98, 0.08); color: #aabd9e; font-size: 12px; font-weight: 500; border: 1px solid rgba(126, 214, 98, 0.2); transition: all 0.2s; }
.green-pill:hover { background: rgba(126, 214, 98, 0.2); color: #fff; border-color: rgba(126, 214, 98, 0.4); cursor: pointer; }

/* --- GRADIO COMPONENT INTEGRATION --- */
.gradio-container .gr-input, .gradio-container .gr-box, .gradio-container textarea, .gradio-container input { background-color: rgba(10, 25, 10, 0.6) !important; border: 1px solid rgba(126, 214, 98, 0.2) !important; color: #fff !important; border-radius: 12px !important; }
.gradio-container .gr-input:focus, .gradio-container textarea:focus { border-color: #30d158 !important; box-shadow: 0 0 0 2px rgba(48, 209, 88, 0.2) !important; }

@media (max-width: 900px) { .grid-4 { grid-template-columns: repeat(2, 1fr); } .grid-2 { grid-template-columns: 1fr; } .nav-row { flex-wrap: wrap; } }


/* --- PLANT SCANNER CUSTOMIZATION --- */
.custom-file { border: none !important; background: transparent !important; }
.custom-file > div { background: rgba(15, 35, 15, 0.4) !important; border: 2px dashed rgba(126, 214, 98, 0.3) !important; border-radius: 20px !important; transition: all 0.3s ease !important; }
.custom-file > div:hover { border-color: #7edf62 !important; background: rgba(126, 214, 98, 0.1) !important; }
.custom-run-btn { background: linear-gradient(135deg, #3a7d2a, #2d5a20) !important; border: none !important; color: white !important; font-weight: 700 !important; font-size: 16px !important; border-radius: 16px !important; padding: 14px !important; box-shadow: 0 8px 25px rgba(36, 110, 36, 0.3) !important; transition: all 0.3s ease !important; margin-top: 15px !important; width: 100% !important; }
.custom-run-btn:hover { background: linear-gradient(135deg, #4e9e38, #3a7d2a) !important; transform: translateY(-2px); box-shadow: 0 10px 30px rgba(48, 209, 88, 0.4) !important; }

/* --- AI SCANNER EXACT MATCH CUSTOMIZATION --- */
.upload-zone { border: none !important; background: transparent !important; margin-bottom: 20px !important; }
.upload-zone > div {
    background: transparent !important;
    border: 1px dashed rgba(126, 214, 98, 0.4) !important;
    border-radius: 20px !important;
    min-height: 280px !important;
    display: flex; align-items: center; justify-content: center;
    transition: all 0.3s ease !important;
}
.upload-zone > div:hover {
    border-color: #7edf62 !important;
    background: rgba(126, 214, 98, 0.05) !important;
}


.small-image-gallery {
    max-height: 240px !important;
    overflow-y: auto !important;
}

.small-image-gallery img {
    max-height: 180px !important;
    object-fit: contain !important;
    border-radius: 12px !important;
}
/* --- PRETTIER MANAGER REMINDERS --- */

.manager-reminder-shell {
    margin-top: 24px;
    margin-bottom: 20px;
}

.manager-reminder-header {
    background:
        radial-gradient(circle at top left, rgba(126,214,98,0.16), transparent 35%),
        linear-gradient(135deg, rgba(12,32,15,0.92), rgba(4,10,6,0.96));
    border: 1px solid rgba(126,214,98,0.22);
    border-radius: 22px;
    padding: 22px;
    box-shadow: 0 16px 40px rgba(0,0,0,0.26);
}

.manager-reminder-form {
    background: rgba(10, 25, 10, 0.55) !important;
    border: 1px solid rgba(126,214,98,0.14) !important;
    border-radius: 18px !important;
    padding: 16px !important;
    margin-bottom: 18px !important;
}

.task-board-row {
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
    padding: 0 !important;
    margin: 12px auto !important;
}

.task-card-pretty {
    width: 100%;
    min-height: 105px;
    display: grid;
    grid-template-columns: 54px minmax(0, 1fr) 115px;
    gap: 16px;
    align-items: center;
    padding: 18px;
    border-radius: 20px;
    background:
        linear-gradient(135deg, rgba(15,35,15,0.90), rgba(6,12,8,0.96));
    border: 1px solid rgba(126,214,98,0.18);
    box-shadow: 0 12px 30px rgba(0,0,0,0.22);
}

.task-number-pill {
    width: 46px;
    height: 46px;
    border-radius: 16px;
    display: flex;
    align-items: center;
    justify-content: center;
    background: rgba(126,214,98,0.12);
    border: 1px solid rgba(126,214,98,0.28);
    color: #d8ffd0;
    font-weight: 900;
    font-size: 16px;
}

.task-cost-box {
    background: rgba(0,0,0,0.24);
    border: 1px solid rgba(126,214,98,0.13);
    border-radius: 15px;
    padding: 10px 12px;
    text-align: right;
}

.task-action-strip {
    display: grid !important;
    grid-template-columns: 1fr 1fr !important;
    gap: 8px !important;
    align-items: center !important;
}

.task-edit-btn button,
.task-delete-btn button {
    min-height: 46px !important;
    border-radius: 14px !important;
    font-weight: 800 !important;
}

.task-edit-btn button {
    background: rgba(126,214,98,0.11) !important;
    color: #d8ffd0 !important;
    border: 1px solid rgba(126,214,98,0.28) !important;
}

.task-delete-btn button {
    background: rgba(239,68,68,0.11) !important;
    color: #ffd6d6 !important;
    border: 1px solid rgba(239,68,68,0.28) !important;
}

/* --- DASHBOARD SPLIT REMINDER LAYOUT --- */

.dashboard-split-shell {
    margin-top: 26px;
    margin-bottom: 26px;
}

.reminder-left-card,
.reminder-right-card {
    background: rgba(15, 35, 15, 0.45) !important;
    border: 1px solid rgba(126, 214, 98, 0.15) !important;
    border-radius: 22px !important;
    padding: 20px !important;
    box-shadow: 0 8px 28px rgba(0, 0, 0, 0.20);
    min-height: 100%;
}

.reminder-form-row {
    background: rgba(10, 25, 10, 0.35) !important;
    border: 1px solid rgba(126, 214, 98, 0.12) !important;
    border-radius: 18px !important;
    padding: 12px !important;
    margin-top: 14px !important;
    margin-bottom: 14px !important;
}

.reminder-task-card {
    background: linear-gradient(135deg, rgba(9, 30, 12, 0.95), rgba(3, 10, 6, 0.96));
    border: 1px solid rgba(126, 214, 98, 0.18);
    border-radius: 18px;
    padding: 16px;
    margin-top: 12px;
    box-shadow: 0 8px 20px rgba(0,0,0,0.16);
}

.reminder-action-row {
    margin-top: 10px !important;
    gap: 8px !important;
}

.reminder-edit-btn button,
.reminder-delete-btn button {
    min-height: 42px !important;
    border-radius: 12px !important;
    font-weight: 800 !important;
}

.reminder-edit-btn button {
    background: rgba(126,214,98,0.10) !important;
    color: #d8ffd0 !important;
    border: 1px solid rgba(126,214,98,0.22) !important;
}

.reminder-delete-btn button {
    background: rgba(239,68,68,0.10) !important;
    color: #ffd6d6 !important;
    border: 1px solid rgba(239,68,68,0.22) !important;
}

.info-mini-card {
    background: rgba(10, 25, 10, 0.42);
    border: 1px solid rgba(126, 214, 98, 0.12);
    border-radius: 16px;
    padding: 14px;
    margin-bottom: 12px;
}

"""


In [10]:
def show_dashboard():
    return (
        gr.update(visible=True),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
    )


def show_upload():
    return (
        gr.update(visible=False),
        gr.update(visible=True),
        gr.update(visible=False),
        gr.update(visible=False),
    )


def show_research():
    return (
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=True),
        gr.update(visible=False),
    )


def show_live():
    return (
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=True),
        make_sensor_cards_html(),
    )


In [11]:
###################
# dashboard_html  #
###################
dashboard_html = """
<div class="main-wrapper">

        <div style="display:grid; grid-template-columns: repeat(3, 1fr); gap:18px; margin-bottom:24px;">

        <div class="page-card">
            <div style="font-size:26px; margin-bottom:8px;">📷</div>
            <div style="color:#aabd9e; font-size:12px; font-weight:800;">PLANT SCANNER</div>
            <div style="color:white; font-size:22px; font-weight:900; margin-top:6px;">Ready</div>
            <div style="color:#30d158; font-size:12px; margin-top:8px;">Image upload available</div>
        </div>

        <div class="page-card">
            <div style="font-size:26px; margin-bottom:8px;">📚</div>
            <div style="color:#aabd9e; font-size:12px; font-weight:800;">ACADEMIC SEARCH</div>
            <div style="color:white; font-size:22px; font-weight:900; margin-top:6px;">Active</div>
            <div style="color:#30d158; font-size:12px; margin-top:8px;">Boolean search enabled</div>
        </div>

        <div class="page-card">
            <div style="font-size:26px; margin-bottom:8px;">📡</div>
            <div style="color:#aabd9e; font-size:12px; font-weight:800;">LIVE TELEMETRY</div>
            <div style="color:white; font-size:22px; font-weight:900; margin-top:6px;">Connected</div>
            <div style="color:#30d158; font-size:12px; margin-top:8px;">Render server data</div>
        </div>

    </div>

</div>
"""


In [12]:
# ==========================================
# MANAGER SETUP / REMINDERS COMPONENT
# ==========================================

import time
import html as html_lib

def render_task_card(task_data, index):
    task_text = html_lib.escape(str(task_data.get("task", "")))
    task_cost = html_lib.escape(str(task_data.get("cost", "")))

    return f"""
    <div class="reminder-task-card">
        <div style="display:flex; justify-content:space-between; align-items:flex-start; gap:14px;">
            <div>
                <div style="color:#9fc99a; font-size:11px; font-weight:850; text-transform:uppercase; letter-spacing:0.08em; margin-bottom:6px;">
                    Reminder #{index + 1}
                </div>
                <div style="color:#ffffff; font-size:18px; font-weight:850; line-height:1.35;">
                    {task_text}
                </div>
            </div>

            <div style="
                min-width:90px;
                text-align:right;
                background: rgba(0,0,0,0.22);
                border: 1px solid rgba(126,214,98,0.12);
                border-radius: 14px;
                padding: 10px 12px;
            ">
                <div style="color:#8ea88a; font-size:10px; font-weight:850; text-transform:uppercase;">
                    Unit cost
                </div>
                <div style="color:#7edf62; font-size:18px; font-weight:900; margin-top:4px;">
                    ₪{task_cost}
                </div>
            </div>
        </div>
    </div>
    """

def build_manager_setup_section():
    tasks_state = gr.State([])

    with gr.Row(elem_classes=["main-wrapper", "dashboard-split-shell"]):

        # ==========================
        # LEFT SIDE - REMINDERS
        # ==========================
        with gr.Column(scale=1, min_width=420):
            with gr.Group(elem_classes=["reminder-left-card"]):

                gr.HTML("""
                <div style="display:flex; align-items:center; gap:12px; margin-bottom:8px;">
                    <div style="
                        width:50px; height:50px; border-radius:16px;
                        display:flex; align-items:center; justify-content:center;
                        background: rgba(126,214,98,0.12);
                        border: 1px solid rgba(126,214,98,0.22);
                        font-size:26px;
                    ">📝</div>

                    <div>
                        <div style="font-size:22px; font-weight:900; color:white;">
                            Plant Care Reminders
                        </div>
                        <div style="font-size:13px; color:#aabd9e; margin-top:4px;">
                            Add daily maintenance tasks for your basil plant.
                        </div>
                    </div>
                </div>
                """)

                with gr.Row(elem_classes=["reminder-form-row"]):
                    reminder_input = gr.Textbox(
                        label="Task / Reminder",
                        placeholder="e.g., Water basil plant",
                        scale=3
                    )

                    unit_cost_input = gr.Number(
                        label="Unit Cost (₪)",
                        value=1.25,
                        scale=1
                    )

                add_task_btn = gr.Button("➕ Add / Save Task", variant="primary")

                manager_message = gr.HTML("")

                def add_new_task(text, cost, current_tasks):
                    current_tasks = list(current_tasks or [])

                    if not text or not text.strip():
                        return (
                            current_tasks,
                            "",
                            "<div style='color:#ffcc00; font-weight:700; margin-top:8px;'>Please enter a task first.</div>"
                        )

                    try:
                        cost = float(cost)
                    except:
                        cost = 0

                    current_tasks.append({
                        "id": int(time.time() * 1000),
                        "task": text.strip(),
                        "cost": cost
                    })

                    return (
                        current_tasks,
                        "",
                        "<div style='color:#30d158; font-weight:700; margin-top:8px;'>Task added successfully.</div>"
                    )

                add_task_btn.click(
                    fn=add_new_task,
                    inputs=[reminder_input, unit_cost_input, tasks_state],
                    outputs=[tasks_state, reminder_input, manager_message]
                )

                @gr.render(inputs=tasks_state)
                def render_dynamic_tasks(tasks_list):
                    if not tasks_list:
                        gr.HTML("""
                        <div style="
                            color:#aabd9e;
                            text-align:center;
                            padding:18px;
                            margin-top:14px;
                            border:1px dashed rgba(126,214,98,0.25);
                            border-radius:14px;
                        ">
                            No reminders yet.
                        </div>
                        """)
                        return

                    for i, task_data in enumerate(tasks_list):
                        gr.HTML(render_task_card(task_data, i))

                        with gr.Row(elem_classes=["reminder-action-row"]):
                            edit_btn = gr.Button("Edit", elem_classes=["reminder-edit-btn"])
                            delete_btn = gr.Button("Delete", elem_classes=["reminder-delete-btn"])

                        def delete_this_task(current_tasks, index=i):
                            current_tasks = list(current_tasks or [])
                            if index < len(current_tasks):
                                current_tasks.pop(index)
                            return (
                                current_tasks,
                                "<div style='color:#30d158; font-weight:700; margin-top:8px;'>Task deleted.</div>"
                            )

                        def edit_this_task(current_tasks, index=i):
                            current_tasks = list(current_tasks or [])
                            if index >= len(current_tasks):
                                return current_tasks, "", 1.25, ""

                            task_to_edit = current_tasks.pop(index)
                            return (
                                current_tasks,
                                task_to_edit["task"],
                                task_to_edit["cost"],
                                "<div style='color:#ffcc00; font-weight:700; margin-top:8px;'>Edit the reminder above and click Add / Save Task.</div>"
                            )

                        delete_btn.click(
                            fn=delete_this_task,
                            inputs=[tasks_state],
                            outputs=[tasks_state, manager_message]
                        )

                        edit_btn.click(
                            fn=edit_this_task,
                            inputs=[tasks_state],
                            outputs=[tasks_state, reminder_input, unit_cost_input, manager_message]
                        )

        # ==========================
        # RIGHT SIDE - INFO PANEL
        # ==========================
        with gr.Column(scale=1, min_width=360):
            with gr.Group(elem_classes=["reminder-right-card"]):
                gr.HTML("""
                <div style="margin-bottom:18px;">
                    <div style="font-size:22px; font-weight:900; color:white; margin-bottom:6px;">
                        About This Dashboard
                    </div>
                    <div style="font-size:14px; color:#aabd9e; line-height:1.6;">
                        This page helps users manage basil plant monitoring in one place.
                        You can review sensor data, upload plant images, search academic information,
                        and maintain daily reminders for plant care.
                    </div>
                </div>

                <div class="info-mini-card">
                    <div style="font-size:15px; font-weight:800; color:white; margin-bottom:5px;">🌱 What can you track?</div>
                    <div style="font-size:13px; color:#aabd9e;">Temperature, humidity, soil moisture, image uploads, and reminder tasks.</div>
                </div>

                <div class="info-mini-card">
                    <div style="font-size:15px; font-weight:800; color:white; margin-bottom:5px;">📌 Reminder examples</div>
                    <div style="font-size:13px; color:#aabd9e;">Water basil, check humidity, inspect leaves, clean sensors, or review growth status.</div>
                </div>

                <div class="info-mini-card">
                    <div style="font-size:15px; font-weight:800; color:white; margin-bottom:5px;">💡 Why is this useful?</div>
                    <div style="font-size:13px; color:#aabd9e;">It helps the user organize routine plant care and estimate maintenance cost over time.</div>
                </div>
                """)


## 5. Frontend Application


In [16]:
with gr.Blocks(css=GLOBAL_CSS,title="My Basil Garden") as app:
    gr.HTML("""
    <div class="main-wrapper">
        <div class="app-header">
            <div class="logo-box">
                <div class="logo-icon">🌿</div>
                <div>
                    <div class="logo-title">My Basil Garden</div>
                    <div class="logo-subtitle">Intelligent Plant Monitoring</div>
                </div>
            </div>
        </div>
    </div>
    """)

    with gr.Row(elem_classes=["nav-row", "main-wrapper"]):
        dashboard_btn = gr.Button("🏠 Dashboard", elem_classes=["nav-btn"])
        upload_btn = gr.Button("📷 Plant Scanner", elem_classes=["nav-btn"])
        research_btn = gr.Button("📚 Academic Search", elem_classes=["nav-btn"])
        live_btn = gr.Button("📡 Live Telemetry", elem_classes=["nav-btn"])

    # Dashboard
    with gr.Group(visible=True) as dashboard_screen:
        gr.HTML(dashboard_html)
        dashboard_sensor_summary = gr.HTML(make_dashboard_sensor_summary_html(fetch_live=False))
        dashboard_refresh_btn = gr.Button("🔄 Refresh Dashboard Sensors")
        build_manager_setup_section()

    # Plant Scanner
    with gr.Group(visible=False) as upload_screen:
        with gr.Column(elem_classes=["main-wrapper"]):
            gr.HTML("""
            <div class="page-card">
                <div class="page-title">Plant Scanner</div>
                <div class="page-subtitle">Upload basil images and request an AI diagnosis.</div>
            </div>
            """)

            plant_images = gr.File(
                label="Upload basil plant images",
                file_count="multiple",
                file_types=["image"],
                elem_classes=["custom-file"],
            )
            upload_status = gr.HTML(update_upload_preview([])[0])
            image_preview = gr.Gallery(
                label="Uploaded Images Preview",
                columns=4,
                rows=1,
                height=180,
                object_fit="contain",
                elem_classes=["small-image-gallery"],
            )
            analyze_btn = gr.Button(
                "🚀 Run AI Analysis",
                variant="primary",
                elem_classes=["custom-run-btn"],
            )
            ai_analysis_output = gr.HTML("")

    # Academic Search
    with gr.Group(visible=False) as research_screen:
        gr.HTML("""
        <div class="main-wrapper" style="margin-bottom:30px; margin-top:10px;">
            <div style="display:flex; align-items:center; gap:16px;">
                <div style="font-size:28px;">🔍</div>
                <div>
                    <div style="font-size:28px; font-weight:800; color:#7edf62;">Academic Search Engine</div>
                    <div style="color:#aabd9e;">Boolean retrieval plus RAG over basil research</div>
                </div>
            </div>
        </div>
        """)

        with gr.Column(elem_classes=["main-wrapper"]):
            with gr.Row(elem_classes=["search-bar-row"]):
                search_text = gr.Textbox(
                    placeholder="Examples: mildew AND basil | essential oil AND antimicrobial",
                    show_label=False,
                    scale=5,
                    lines=2,
                )
                search_btn = gr.Button("Search", variant="primary", scale=1)

            search_output = gr.HTML("""
            <div class="page-card" style="text-align:center; padding:60px 20px;">
                <div style="font-size:64px;">📚</div>
                <h3 style="color:white;">Ready to explore</h3>
                <p style="color:#aabd9e;">Search the indexed academic database.</p>
            </div>
            """)

    # Live Telemetry
    with gr.Group(visible=False) as live_screen:
        sensor_cards = gr.HTML(make_sensor_cards_html(fetch_live=False))

        with gr.Column(elem_classes=["main-wrapper"]):
            with gr.Row():
                temperature_btn = gr.Button("🌡️ Temperature")
                humidity_btn = gr.Button("💧 Humidity")
                soil_btn = gr.Button("🌱 Soil")
                refresh_cards_btn = gr.Button("🔄 Refresh Cards", variant="primary")

            sensor_limit = gr.Slider(1, 100, value=10, step=1, label="Samples")
            live_output = gr.HTML("""
            <div class="page-card" style="text-align:center;">
                <h3 style="color:white;">Choose a sensor</h3>
                <p style="color:#aabd9e;">The graph and table will appear here.</p>
            </div>
            """)
            live_plot = gr.Plot(label="Sensor Graph")
            live_table = gr.Dataframe(label="Sensor Data Table")

    # Navigation wiring
    dashboard_btn.click(
        show_dashboard,
        outputs=[dashboard_screen, upload_screen, research_screen, live_screen],
    )
    upload_btn.click(
        show_upload,
        outputs=[dashboard_screen, upload_screen, research_screen, live_screen],
    )
    research_btn.click(
        show_research,
        outputs=[dashboard_screen, upload_screen, research_screen, live_screen],
    )
    live_btn.click(
        show_live,
        outputs=[dashboard_screen, upload_screen, research_screen, live_screen, sensor_cards],
    )

    # Scanner wiring
    plant_images.change(
        update_upload_preview,
        inputs=[plant_images],
        outputs=[upload_status, image_preview],
    )
    analyze_btn.click(
        run_gemini_analysis,
        inputs=[plant_images],
        outputs=[ai_analysis_output],
    )

    # Search wiring
    search_btn.click(search_and_wrap, inputs=[search_text], outputs=[search_output])
    search_text.submit(search_and_wrap, inputs=[search_text], outputs=[search_output])

    # Sensor wiring
    temperature_btn.click(
        show_temperature_graph,
        inputs=[sensor_limit],
        outputs=[sensor_cards, live_output, live_plot, live_table],
    )
    humidity_btn.click(
        show_humidity_graph,
        inputs=[sensor_limit],
        outputs=[sensor_cards, live_output, live_plot, live_table],
    )
    soil_btn.click(
        show_soil_graph,
        inputs=[sensor_limit],
        outputs=[sensor_cards, live_output, live_plot, live_table],
    )
    refresh_cards_btn.click(make_sensor_cards_html, outputs=[sensor_cards])
    dashboard_refresh_btn.click(
        make_dashboard_sensor_summary_html,
        outputs=[dashboard_sensor_summary],
    )

print("Frontend application constructed successfully.")


/tmp/ipykernel_2075/2739943644.py:1: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=GLOBAL_CSS,title="My Basil Garden") as app:
/tmp/ipykernel_2075/263494690.py:125: DeprecationWarning: The 'show_api' parameter in event listeners will be removed in Gradio 6.0. You will need to use the 'api_visibility' parameter instead. To replicate show_api=False, in Gradio 6.0, use api_visibility='undocumented'.
  @gr.render(inputs=tasks_state)


Frontend application constructed successfully.


## 6. QA Tests


In [14]:
def run_qa_tests():
    tests = []

    def check(name: str, condition: bool):
        if not condition:
            raise AssertionError(name)
        tests.append(name)

    check("Five basil papers are loaded", len(basil_papers) == 5)
    check("Target inverted index is not empty", bool(inverted_index))
    check("Mildew is indexed", "mildew" in inverted_index)
    check(
        "Boolean AND returns only matching papers",
        all(
            {"mildew", "basil"}.issubset(set(result["found_words"]))
            for result in boolean_search("mildew AND basil")
        ),
    )
    check("Boolean OR returns results", len(boolean_search("mildew OR linalool")) > 0)

    local_rag = EcologicalRAG(gemini_api_key=None, use_transformers=False)
    local_rag.load_papers(basil_papers)
    rag_result = local_rag.query("What causes basil downy mildew?", n_results=2)
    check("RAG returns two papers", rag_result["papers_found"] == 2)
    check("RAG response is text", isinstance(rag_result["response"], str))

    check("37.5°C is correctly marked High", sensor_status("temperature", 37.5)[0] == "High")
    check("25°C is correctly marked Optimal", sensor_status("temperature", 25)[0] == "Optimal")
    check("100% soil moisture is Too Wet", sensor_status("soil", 100)[0] == "Too Wet")
    check("Missing sensor value is No Data", sensor_status("humidity", "--")[0] == "No Data")

    empty_status, empty_gallery = update_upload_preview(None)
    check("Empty upload preview is handled", "No images uploaded" in empty_status and empty_gallery == [])

    print(f"✅ {len(tests)} QA tests passed:")
    for test in tests:
        print("  -", test)


run_qa_tests()


✅ 12 QA tests passed:
  - Five basil papers are loaded
  - Target inverted index is not empty
  - Mildew is indexed
  - Boolean AND returns only matching papers
  - Boolean OR returns results
  - RAG returns two papers
  - RAG response is text
  - 37.5°C is correctly marked High
  - 25°C is correctly marked Optimal
  - 100% soil moisture is Too Wet
  - Missing sensor value is No Data
  - Empty upload preview is handled


## 7. Launch


In [17]:
# Run this cell only after all QA tests pass.
app.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f42a7882027887a22f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
